# Day 3: Baseline Model & XGBoost
Train/test split, scaling, SMOTE for class imbalance, Logistic Regression baseline, XGBoost.


In [1]:
import numpy as np
import pandas as pd 
from pathlib import Path

DATA_PATH = Path("../data/processed/telco_churn_processed.csv")
df = pd.read_csv(DATA_PATH)
df.shape

(7043, 28)

In [2]:
from sklearn.model_selection import train_test_split 
X = df.drop(columns = ["Churn"])
y = df["Churn"]

X_train, X_test, y_train,y_test = train_test_split(X, y, test_size = 0.2, stratify = y, random_state = 42)

print("Train shape", X_train.shape, "| Teat shape:", X_test.shape)
print("Train churn rate:", y_train.mean().round(4))
print("Test churn rate:", y_test.mean().round(4))

Train shape (5634, 27) | Teat shape: (1409, 27)
Train churn rate: 0.2654
Test churn rate: 0.2654


In [3]:
from sklearn.preprocessing import StandardScaler
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges", "TotalServices"]
scaler = StandardScaler()

X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

X_train[numeric_cols].describe()

,tenure,MonthlyCharges,TotalCharges,TotalServices
count,5.634000e+03,5.634000e+03,5.634000e+03,5634.000000
mean,-1.008935e-17,-2.402527e-16,2.522338e-17,0.000000
std,1.000089e+00,1.000089e+00,1.000089e+00,1.000089
min,-1.322329e+00,-1.544028e+00,-1.008922e+00,-1.109762
25%,-9.559779e-01,-9.711977e-01,-8.321009e-01,-1.109762
50%,-1.418632e-01,1.848336e-01,-3.968446e-01,-0.031297
75%,9.164859e-01,8.319124e-01,6.741944e-01,0.507935
max,1.608483e+00,1.785939e+00,2.801869e+00,2.125633


In [7]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state = 42)
X_train_smote, y_train_smote= smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE:", y_train_smote.value_counts().to_dict())

Before SMOTE: {0: 4139, 1: 1495}
After SMOTE: {0: 4139, 1: 4139}


In [8]:
#Training Logistic Regression Baseline

from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(random_state = 42, max_iter = 1000)
log_reg.fit(X_train_smote, y_train_smote)

y_pred_log = log_reg.predict(X_test)
y_pred_log[:10]

array([0, 1, 0, 1, 0, 1, 0, 0, 0, 0])

In [9]:
#checking accuracy
from sklearn.metrics import accuracy_score
acc =accuracy_score(y_test, y_pred_log)
print("Logistic Regression accuracy:", round(acc,4))

Logistic Regression accuracy: 0.7622


In [14]:
#Train XGBoost

from xgboost import XGBClassifier
xgb_model = XGBClassifier(random_state = 42, eval_metric = "logloss")
xgb_model.fit(X_train_smote, y_train_smote)

y_pred_xgb = xgb_model.predict(X_test)
acc_xgb = accuracy_score(y_test, y_pred_xgb)
print("XGBoost Accuracy:", round(acc_xgb, 4))

XGBoost Accuracy: 0.763


In [15]:
import joblib
from pathlib import Path

MODELS_DIR = Path("../outputs/models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(log_reg, "../outputs/models/logistic_regression.joblib")
joblib.dump(xgb_model, "../outputs/models/xgboost.joblib")
joblib.dump(scaler,"../outputs/models/scaler.joblib")
print("Models saved.")

Models saved.
